# Module 4 - Performance, cost, Lakeflow Jobs and DABs

![Query profile reading map](../assets/images/query_profile_reading_map.png)

This is production-readiness orientation. We do not build a full CI/CD lab, but
we show what should be automated and how it can be described as code.

In [ ]:
%run ../setup/00_setup

## Runtime pre-check

Run Modules 2 and 3 first. This module compares detail vs aggregate query
patterns and references the BI-ready objects.

In [ ]:
required_objects = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
]

missing = [table for table in required_objects if not spark.catalog.tableExists(table)]
if missing:
    raise Exception("Missing objects. Run Modules 2 and 3 first: " + ", ".join(missing))

print("[OK] Performance demo objects exist")

## Before/after performance

Bad pattern: DirectQuery report scans detailed Silver rows on every interaction.
Better pattern: query Gold aggregate for summary pages and detail view only for
drill-through.

In [ ]:
spark.sql(f"""
EXPLAIN FORMATTED
SELECT category, SUM(line_revenue) AS revenue
FROM {SILVER}.order_lines
WHERE status = 'Completed'
GROUP BY category
""").show(80, truncate=False)

In [ ]:
spark.sql(f"""
EXPLAIN FORMATTED
SELECT category, SUM(revenue) AS revenue
FROM {GOLD}.fact_sales_dashboard_monthly
GROUP BY category
""").show(80, truncate=False)

## Query profile reading

Use the map at the top of this notebook with the real Databricks Query Profile.
The training goal is to connect what participants see in the plan to an action:

- large scan -> Gold aggregate or narrower columns,
- shuffle -> pre-aggregation or different grain,
- slow join -> star model or BI-ready table,
- many tiny files -> OPTIMIZE / predictive optimization,
- repeated DirectQuery -> Import or cache-friendly aggregate.

## Practical optimization checklist

| Technique | Training message |
|---|---|
| Column pruning | do not feed Power BI `SELECT *` sources |
| Predicate pushdown | date filters must fold to Databricks |
| Monthly aggregate | summary visuals should not scan line grain |
| OPTIMIZE / stats | physical layout helps, but does not replace good modelling |
| Warehouse sizing | size is a trade-off between runtime and DBU/h |

## Lakeflow Job DAG

![Lakeflow Job DAG](../assets/images/lakeflow_job_dag.png)

## DABs deployment flow

![DABs deployment flow](../assets/images/dabs_deployment_flow.png)

Reference file: `bundle/databricks.yml`.

## Automation readiness checklist

![Automation readiness checklist](../assets/images/automation_readiness_checklist.png)

## Refresh strategy

Use `docs/templates/refresh-strategy.md`.

Recommended DAG:

1. Validate source data.
2. Refresh Gold model.
3. Reconcile KPI.
4. Build BI aggregate.
5. Publish BI readiness status.

Important operational ideas:

- schedule for daily reports,
- table update trigger for upstream refresh,
- repair run after a failed transform,
- job parameters for dev/test/prod,
- owner notification when certification fails.

## Cost guardrails

Use `docs/templates/cost-awareness-checklist.md`.

The key training message: live BI is not free. It changes the warehouse usage
pattern from scheduled refresh to interactive query fan-out.

## Long-form delivery extension

If you need to stretch toward 9 hours, use these extra prompts:

- compare the two `EXPLAIN FORMATTED` plans line by line,
- ask participants to decide which DAG task should fail-fast,
- convert one workshop decision into a `bundle/databricks.yml` parameter,
- sketch dev/test/prod promotion on the DABs diagram,
- discuss how Power BI live pages affect SQL Warehouse auto-stop and concurrency.